In [ ]:
from google.colab import files
uploaded = files.upload()

Saving LFS Guinea bissau (2) (1).dta to LFS Guinea bissau (2) (1).dta


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_name = list(uploaded.keys())[0]

df = pd.read_stata(file_name, convert_categoricals=False)

print("Full LFS shape:", df.shape)
df.head()

/tmp/ipykernel_2070/908034988.py:7: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata(file_name, convert_categoricals=False)


Full LFS shape: (30400, 978)


,STRATE,hh1,hh2,m1,SE1_1,SE1_2,SE1_3,SE1_4,SE1_5,SE1_6,...,secteur1_as1,secteur1_as2,secteur2_ap,secteur2_as1,secteur2_as2,typemp_ap,REGIAO,milieu_etendu,weightemploi,Idademaior15
0,52,163,0,1,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,14,3,9.853382,0
1,71,225,12,1,0.0,0.0,2.0,0.0,0.0,0.0,...,NaN,NaN,2.0,NaN,NaN,2.0,12,2,59.853987,1
2,71,225,12,2,0.0,1.0,1.0,1.0,1.0,0.0,...,NaN,NaN,1.0,NaN,NaN,2.0,12,2,59.853987,1
3,71,225,12,4,25.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,12,2,59.853987,0
4,71,225,12,5,25.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,12,2,59.853987,0


In [ ]:
key_vars = ["TP4_SECT", "TP13_BRAN", "sexe"]

for col in key_vars:
    print("\n---", col, "---")
    print(df[col].value_counts(dropna=False).head(30))


--- TP4_SECT ---
TP4_SECT
NaN    17759
1.0     8085
4.0     3488
3.0      853
2.0      215
Name: count, dtype: int64

--- TP13_BRAN ---
TP13_BRAN
NaN       27696
111.0       399
113.0       192
121.0       148
4120.0      117
150.0       116
4736.0      101
4721.0       93
4731.0       89
4710.0       87
9700.0       60
136.0        59
8540.0       53
301.0        53
8102.0       51
4735.0       47
4921.0       41
9801.0       41
1400.0       27
8420.0       26
4723.0       25
4623.0       24
8000.0       23
8203.0       23
230.0        21
8202.0       19
1622.0       19
1022.0       19
8510.0       18
9802.0       18
Name: count, dtype: int64

--- sexe ---
sexe
1.0    15759
0.0    14593
NaN       48
Name: count, dtype: int64


In [ ]:
df_work = df[df["TP4_SECT"].notna()].copy()

print("Working population shape:", df_work.shape)
df_work["TP4_SECT"].value_counts(dropna=False)

Working population shape: (12641, 978)


,count
TP4_SECT,
1.0,8085
4.0,3488
3.0,853
2.0,215


In [ ]:
# Confirmed from your LFS:
# sexe = 1 Male
# sexe = 0 Female

df_work["gender"] = df_work["sexe"].map({
    1: "Male",
    0: "Female"
})

df_work["gender"].value_counts(dropna=False)

,count
gender,
Male,6407
Female,6208
NaN,26


In [ ]:
def classify_agri_food(row):
    sect = row["TP4_SECT"]
    bran = row["TP13_BRAN"]

    # 1. Primary agriculture
    # TP4_SECT == 1 captures agriculture workers
    if sect == 1:
        return "Primary agriculture"

    # If no branch code, cannot classify as agribusiness
    if pd.isna(bran):
        return "Non-agriculture"

    code = int(bran)

    # 2. Food manufacturing / agro-processing
    # Include ISIC Division 10 only: 1000–1099
    # Exclude tobacco completely
    if 1000 <= code < 1100:
        return "Food manufacturing"

    # 3. Distribution / trade
    # Based on your ISIC framework
    elif code in [4620, 4630, 4721, 4781]:
        return "Distribution"

    # 4. Agri-services / logistics
    # Transport and storage
    elif (4900 <= code < 5000) or code == 5210:
        return "Agri-logistic"

    else:
        return "Non-agriculture"


df_work["agri_value_chain"] = df_work.apply(classify_agri_food, axis=1)

df_work["agri_value_chain"].value_counts(dropna=False)

,count
agri_value_chain,
Primary agriculture,8085
Non-agriculture,4494
Distribution,24
Agri-logistic,20
Food manufacturing,18


In [ ]:
df_agri = df_work[
    df_work["agri_value_chain"].isin([
        "Primary agriculture",
        "Food manufacturing",
        "Distribution",
        "Agri-logistic"
    ])
].copy()

print("Agri-food dataset shape:", df_agri.shape)
df_agri["agri_value_chain"].value_counts()

Agri-food dataset shape: (8147, 980)


,count
agri_value_chain,
Primary agriculture,8085
Distribution,24
Agri-logistic,20
Food manufacturing,18


In [ ]:
summary = df_agri["agri_value_chain"].value_counts().reset_index()
summary.columns = ["agri_value_chain", "workers"]

summary["share_%"] = summary["workers"] / summary["workers"].sum() * 100

summary

,agri_value_chain,workers,share_%
0,Primary agriculture,8085,99.238984
1,Distribution,24,0.294587
2,Agri-logistic,20,0.245489
3,Food manufacturing,18,0.220940


In [ ]:
gender_summary = pd.crosstab(
    df_agri["agri_value_chain"],
    df_agri["gender"],
    margins=True
)

gender_summary

gender,Female,Male,All
agri_value_chain,,,
Agri-logistic,19,1,20
Distribution,6,18,24
Food manufacturing,9,9,18
Primary agriculture,3976,4096,8072
All,4010,4124,8134


In [ ]:
gender_share = pd.crosstab(
    df_agri["agri_value_chain"],
    df_agri["gender"],
    normalize="index"
) * 100

gender_share.round(1)

gender,Female,Male
agri_value_chain,,
Agri-logistic,95.0,5.0
Distribution,25.0,75.0
Food manufacturing,50.0,50.0
Primary agriculture,49.3,50.7


In [ ]:
df_stata = df_agri.copy()

# Make column names Stata-safe
df_stata.columns = [
    c[:30].replace(" ", "_").replace("-", "_")
    for c in df_stata.columns
]

df_stata.to_stata(
    "lfs_agriculture_agribusiness_clean.dta",
    write_index=False
)

files.download("lfs_agriculture_agribusiness_clean.dta")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_agri.to_csv("lfs_agriculture_agribusiness_clean.csv", index=False)
df_agri.to_excel("lfs_agriculture_agribusiness_clean.xlsx", index=False)

files.download("lfs_agriculture_agribusiness_clean.csv")
files.download("lfs_agriculture_agribusiness_clean.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>